# Charlie Eden

4/10/2026

In [1]:
# Setting working dir
import os
from pathlib import Path
p = os.getcwd() + "/../../"
os.chdir(p)
parent_dir = os.getcwd()

In [3]:
import numpy as np

config = {
    "years": np.arange(2019, 2025),
    "imd_folder": f"{parent_dir}/../monsoon-benchmark_data/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thresh_file": f"{parent_dir}/../monsoon-benchmark_data/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{parent_dir}/../monsoon-benchmark_data/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{parent_dir}/examples/paper_figures/outputs",  # Directory to save data files,
    "mok": True,
    "day_bins_30": [(1, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)],
    "day_bins_15": [(1, 5), (6, 10), (11, 15)],
    "mem_num": 51,
    "date_filter_year": 2024,
    "file_pattern": "{}.nc"
}

# Paths to 4p0 model forecast data (.nc)
model_paths = {
    "IFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/IFS_S2S",  # IFS model
    "AIFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/AIFS",  # AIFS model
    "FuXi": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi",  # FuXi mdoel
    "Graphcast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GraphCast",  # Graphcast model
    "GenCast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GenCast",  # GenCast model
    "FuXi-S2S": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S",  # FuXi_S2S model
    "NGCM": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/NeuralGCM",  # NGCM model
}

In [4]:
from monsoonbench.metrics import ProbabilisticOnsetMetrics
from monsoonbench.metrics import ClimatologyOnsetMetrics
from monsoonbench.visualization.compare_models import calculate_reliability_metrics
import xarray as xr

c = ClimatologyOnsetMetrics()
p = ProbabilisticOnsetMetrics()

skills_15 = {}
skills_30 = {}
brier_15 = {}
brier_30 = {}
auc_15 = {}
auc_30 = {}
reliability_15 = {}

thresh_ds = xr.open_dataset(config["thresh_file"])
thresh_slice = thresh_ds["MWmean"]
clim_onset = c.compute_climatological_onset_dataset(
            config["imd_folder"], thresh_slice, years=None, mok=config["mok"]
        )

Processing 124 years: [1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Processing year 1901...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd)

In [5]:
# 15 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            range(2019, 2024),
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=15,
            day_bins=config["day_bins_15"],
            date_filter_year=config["date_filter_year"],
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["years"],
            config["day_bins_15"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=config["date_filter_year"],
            file_pattern=config["file_pattern"],
            max_forecast_day=15,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    reliability_metrics = calculate_reliability_metrics(forecast_df)

    skills_15[model_name] = skill_results
    reliability_15[model_name] = reliability_metrics
    auc_15[model_name] = auc_forecast
    brier_15[model_name] = brier_forecast
        



Processing IFS
Processing years: range(2019, 2024)


Using 4-degree CMZ polygon coordinates

Processing year 2019
Loading S2S model data...
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2019.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 26 init times x 10 unique locations x 11 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0)), (np.float64(76.0), np.float64(28.0)), (np.float64(80.0), np.float64(28.0))]
Using MOK (June 2nd filter) for onset detection


Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_obs_pairs
    ProbabilisticOnsetMetrics.get_forecast_probabilistic_twice_weekly_2(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        year,
        ^^^^^
    ...<3 lines>...
        file_pattern,
        ^^^^^^^^^^^^^
    )
    ^
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 120, in get_forecast_probabilistic_twice_weekly_2
    raise FileNotFoundError(f"File not found: {file_path}")
FileNotFoundError: File not found: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S/2022.nc
Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_

Generated 390 climatological forecast-observation pairs
Unique lat-lon pairs processed: 10
Total bins per forecast: 5
Probability range: 0.000 - 1.000
Observed onset rate: 0.200
Non-zero probabilities: 228
Unique locations in output: 10

Distribution across bins:
                      predicted_prob        observed_onset  \
                               count   mean           mean   
bin_label                                                    
After day 15                      78  0.677          0.731   
Before initialization             78  0.108          0.000   
Days 1-5                          78  0.065          0.077   
Days 11-15                        78  0.082          0.103   
Days 6-10                         78  0.067          0.090   

                      n_contributing_years total_members_with_onset  
                                      mean                     mean  
bin_label                                                            
After day 15                 

In [6]:
# 30 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            range(2019, 2024),
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=30,
            day_bins=config["day_bins_30"],
            date_filter_year=config["date_filter_year"],
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["years"],
            config["day_bins_30"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=config["date_filter_year"],
            file_pattern=config["file_pattern"],
            max_forecast_day=30,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    skills_30[model_name] = skill_results  
    auc_30[model_name] = auc_forecast
    brier_30[model_name] = brier_forecast      



Processing IFS
Processing years: range(2019, 2024)
Using 4-degree CMZ polygon coordinates

Processing year 2019
Loading S2S model data...
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2019.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 26 init times x 10 unique locations x 11 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0)), (np.float64(76.0), np.float64(28.0)), (np.float64(80.0), np.float64(28.

Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_obs_pairs
    ProbabilisticOnsetMetrics.get_forecast_probabilistic_twice_weekly_2(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        year,
        ^^^^^
    ...<3 lines>...
        file_pattern,
        ^^^^^^^^^^^^^
    )
    ^
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 120, in get_forecast_probabilistic_twice_weekly_2
    raise FileNotFoundError(f"File not found: {file_path}")
FileNotFoundError: File not found: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S/2022.nc
Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_

Generated 624 climatological forecast-observation pairs
Unique lat-lon pairs processed: 10
Total bins per forecast: 8
Probability range: 0.000 - 1.000
Observed onset rate: 0.125
Non-zero probabilities: 402
Unique locations in output: 10

Distribution across bins:
                      predicted_prob        observed_onset  \
                               count   mean           mean   
bin_label                                                    
After day 30                      78  0.417          0.462   
Before initialization             78  0.108          0.000   
Days 1-5                          78  0.065          0.077   
Days 11-15                        78  0.082          0.103   
Days 16-20                        78  0.088          0.077   
Days 21-25                        78  0.086          0.115   
Days 26-30                        78  0.086          0.077   
Days 6-10                         78  0.067          0.090   

                      n_contributing_years total_memb

In [9]:
import pandas as pd
rows = []
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 15

    row["fair_brier_skill_d1_5"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = None
    row["fair_brier_skill_d21_25"] = None
    row["fair_brier_skill_d26_30"] = None

    row["fair_brier_d1_5"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = None
    row["fair_brier_d21_25"] = None
    row["fair_brier_d26_30"] = None

    row["auc_d1_5"] = auc_15[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = auc_15[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = auc_15[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = None
    row["auc_d21_25"] = None
    row["auc_d26_30"] = None
    row["auc_later"] = auc_15[model_name]["bin_auc_scores"]["After day 15"]

    rows.append(row)


    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 30

    row["fair_brier_skill_d1_5"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 16-20"]
    row["fair_brier_skill_d21_25"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 21-25"]
    row["fair_brier_skill_d26_30"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 26-30"]

    row["fair_brier_d1_5"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 16-20"]
    row["fair_brier_d21_25"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 21-25"]
    row["fair_brier_d26_30"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 26-30"]

    row["auc_d1_5"] = auc_30[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = auc_30[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = auc_30[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = auc_30[model_name]["bin_auc_scores"]["Days 16-20"]
    row["auc_d21_25"] = auc_30[model_name]["bin_auc_scores"]["Days 21-25"]
    row["auc_d26_30"] = auc_30[model_name]["bin_auc_scores"]["Days 26-30"]
    row["auc_later"] = auc_30[model_name]["bin_auc_scores"]["After day 30"]

    rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df


,model_label,horizon,fair_brier_skill_d1_5,fair_brier_skill_d6_10,fair_brier_skill_d11_15,fair_brier_skill_d16_20,fair_brier_skill_d21_25,fair_brier_skill_d26_30,fair_brier_d1_5,fair_brier_d6_10,...,fair_brier_d16_20,fair_brier_d21_25,fair_brier_d26_30,auc_d1_5,auc_d6_10,auc_d11_15,auc_d16_20,auc_d21_25,auc_d26_30,auc_later
0,ifs,15,0.397846,0.147049,0.046467,NaN,NaN,NaN,0.048682,0.068558,...,NaN,NaN,NaN,0.922698,0.851670,0.771822,NaN,NaN,NaN,0.927304
1,ifs,30,0.397846,0.147049,0.046467,0.049100,-0.001361,0.013516,0.048682,0.068558,...,0.079129,0.087224,0.086659,0.922698,0.851670,0.771822,0.766231,0.694732,0.721722,0.895437
2,gencast,15,0.332756,0.065693,0.052427,NaN,NaN,NaN,0.052584,0.075229,...,NaN,NaN,NaN,0.929903,0.833178,0.816743,NaN,NaN,NaN,0.929464
3,gencast,30,0.332756,0.065693,0.052427,-0.052475,-0.044604,-0.027344,0.052584,0.075229,...,0.084940,0.090560,0.088514,0.929903,0.833178,0.816743,0.774316,0.728810,0.731957,0.900925
4,fuxi-s2s,15,0.208563,0.069433,-0.155639,NaN,NaN,NaN,0.061938,0.067932,...,NaN,NaN,NaN,0.808628,0.871501,0.661179,NaN,NaN,NaN,0.872859
5,fuxi-s2s,30,0.208563,0.069433,-0.155639,-0.081604,-0.272543,-0.018465,0.061938,0.067932,...,0.094841,0.110850,0.078090,0.808628,0.871501,0.661179,0.748228,0.651966,0.774356,0.900549
6,ngcm,15,0.334993,0.124527,0.055870,NaN,NaN,NaN,0.052407,0.070492,...,NaN,NaN,NaN,0.947398,0.875928,0.824400,NaN,NaN,NaN,0.938979
7,ngcm,30,0.334993,0.124527,0.055870,-0.005738,-0.047950,-0.092803,0.052407,0.070492,...,0.081168,0.090850,0.094154,0.947398,0.875928,0.824400,0.812459,0.725694,0.682558,0.908256


In [8]:
# Reading in Matlab files

from scipy.io import loadmat

def load_mat_to_dict(file_path:str,
                    vars_of_interest: list = None):
    data = loadmat(file_path)
    if vars_of_interest:
        ret_dict = {var: data[var] for var in vars_of_interest}
        return ret_dict
    return data